# Scheduler RL Minimal Colab (OpenEnv + TRL/Unsloth)

This notebook gives you a minimal, hackathon-ready training/evaluation flow for your scheduler environment.

In [ ]:
!pip -q install "openenv-core[core]>=0.2.2" trl unsloth transformers datasets accelerate matplotlib pandas requests

In [ ]:
# Optional connectivity check (run once if needed)
# import requests
# r = requests.get("http://127.0.0.1:8000/session/live_tracks", timeout=10)
# print("status:", r.status_code)
# print("json keys:", list(r.json().keys()))
# print("count:", r.json().get("count"))

## Configure Base URL

- Local run: `http://127.0.0.1:8000`
- HF Space run: `https://VAKYA-Report_generation.hf.space`

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "http://127.0.0.1:8000"
EPISODES = 20

def reset_tracks():
    requests.post(f"{BASE_URL}/session/live_tracks/reset", timeout=30)

def run_schedule(slot="both"):
    r = requests.post(f"{BASE_URL}/session/run_manual_schedule", json={"slot": slot}, timeout=120)
    r.raise_for_status()
    return r.json()

def fetch_tracks():
    r = requests.get(f"{BASE_URL}/session/live_tracks", timeout=30)
    r.raise_for_status()
    return r.json().get("rows", [])

def episode_reward(rows):
    total = len(rows)
    if total == 0:
        return {"total": 0, "success": 0, "failed": 0, "avg_retries": 0.0, "reward": 0.0}
    success = sum(1 for r in rows if r.get("status") == "success")
    failed = sum(1 for r in rows if r.get("status") == "failed")
    avg_retries = sum(float(r.get("retries_used", 0)) for r in rows) / total
    success_rate = success / total
    reward = (10.0 * success_rate) - (2.5 * (failed / total)) - (0.2 * avg_retries)
    return {
        "total": total,
        "success": success,
        "failed": failed,
        "avg_retries": avg_retries,
        "reward": reward,
    }

In [ ]:
# Optional quick API ping
# import requests
# print(requests.get("http://127.0.0.1:8000/session/live_tracks", timeout=10).status_code)
# print(requests.get("http://127.0.0.1:8000/session/live_tracks", timeout=10).json().keys())

In [ ]:
# Baseline random policy over scheduler actions
import random

rows = []
for ep in range(1, EPISODES + 1):
    reset_tracks()
    slot = random.choice(["10am", "11am", "both"])
    run_schedule(slot)
    m = episode_reward(fetch_tracks())
    rows.append({"episode": ep, "policy": "random", "slot": slot, **m})

df_random = pd.DataFrame(rows)
df_random.tail()

In [ ]:
# Strong hand-crafted policy baseline (always run both)
rows = []
for ep in range(1, EPISODES + 1):
    reset_tracks()
    run_schedule("both")
    m = episode_reward(fetch_tracks())
    rows.append({"episode": ep, "policy": "both_only", "slot": "both", **m})

df_both = pd.DataFrame(rows)
df_both.tail()

In [ ]:
# Plot reward curves for pitch/demo
plt.figure(figsize=(9,4))
plt.plot(df_random["episode"], df_random["reward"], label="random policy")
plt.plot(df_both["episode"], df_both["reward"], label="both-only policy")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Scheduler Environment Reward Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("hackathon/reward_curve.png", dpi=160)
plt.show()

print("Saved: hackathon/reward_curve.png")
print("Random mean reward:", round(df_random["reward"].mean(), 4))
print("Both-only mean reward:", round(df_both["reward"].mean(), 4))

## TRL + Unsloth GRPO Minimal Training Run

This section runs a minimal RL training loop where the model outputs one scheduler action token (`10am`, `11am`, or `both`) and reward is computed directly from your live environment.

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompts = [
    {"prompt": "Choose best scheduler action for enterprise reporting. Respond only: 10am or 11am or both."}
    for _ in range(64)
]
ds = Dataset.from_list(prompts)

def slot_from_text(text: str) -> str:
    text = text.lower()
    if "both" in text:
        return "both"
    if "11am" in text:
        return "11am"
    return "10am"

def reward_fn(completions, **kwargs):
    rewards = []
    for c in completions:
        slot = slot_from_text(c)
        reset_tracks()
        run_schedule(slot)
        m = episode_reward(fetch_tracks())
        rewards.append(float(m["reward"]))
    return rewards

cfg = GRPOConfig(
    output_dir="grpo_scheduler_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,
    max_prompt_length=128,
    max_completion_length=16,
    num_train_epochs=1,
    logging_steps=1,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=cfg,
    train_dataset=ds,
)

# Minimal real training run (required for hackathon evidence)
train_output = trainer.train()
train_output

In [ ]:
# Save training artifacts (CSV + PNG) for README/judging evidence
import os

os.makedirs("hackathon", exist_ok=True)

# Save baseline comparison CSV from earlier cells
df_compare = pd.concat([df_random, df_both], ignore_index=True)
df_compare.to_csv("hackathon/baseline_policy_rewards.csv", index=False)

# Try to extract trainer logs for loss/reward trends
log_history = getattr(getattr(trainer, "state", None), "log_history", None)
if log_history:
    df_logs = pd.DataFrame(log_history)
    df_logs.to_csv("hackathon/grpo_train_logs.csv", index=False)

    if "loss" in df_logs.columns and "step" in df_logs.columns:
        plt.figure(figsize=(8, 4))
        plt.plot(df_logs["step"], df_logs["loss"], label="train_loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("GRPO Training Loss")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig("hackathon/grpo_loss_curve.png", dpi=160)
        plt.show()

print("Saved: hackathon/baseline_policy_rewards.csv")
if log_history:
    print("Saved: hackathon/grpo_train_logs.csv")
    if "loss" in df_logs.columns and "step" in df_logs.columns:
        print("Saved: hackathon/grpo_loss_curve.png")